# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.496500,0.527291,0.637203,0.806670,...,0.3,0.815075,0.788450,0.867595,0.911292,0.783514,0.868510,0.784140,0.618609,0.543036
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.340099,0.303278,0.512236,0.734278,...,0.7,0.686883,0.686778,0.594025,0.698730,0.695484,0.772514,0.666212,0.572044,0.446302
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.127990,0.192701,0.330117,0.533609,...,0.5,0.580053,0.607686,0.523752,0.600009,0.614942,0.717012,0.570198,0.631260,0.440385
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.249994,-1.823374,-0.834575,0.001354,...,0.0,0.002635,0.002627,0.013206,0.025528,0.000000,0.005494,0.063219,0.000000,0.076485
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.090382,-0.011299,0.200067,0.386963,...,0.3,0.397511,0.457883,0.364956,0.265409,0.452112,0.482416,0.418672,0.245866,0.244423


In [5]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,-1.721325,-0.872993,0.005473,...,0.0,0.006534,0.003712,0.009642,0.010313,0.000000,0.019849,0.061327,0.000000,0.076454
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,-0.919863,-0.289991,0.054146,...,0.1,0.063555,0.110500,0.044563,0.145672,0.047794,0.012173,0.144461,0.102869,0.089535
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,-1.809246,-0.917667,0.003687,...,0.0,0.004404,0.003123,0.008417,0.006856,0.000000,0.010304,0.057929,0.000000,0.076454
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.328363,0.387739,0.676445,...,0.3,0.670446,0.712745,0.648495,0.504756,0.637237,0.444558,0.628242,0.183451,0.215908
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.644014,0.670147,0.852891,...,0.8,0.835307,0.855796,0.796202,0.823601,0.787922,0.939648,0.787450,0.823541,0.643512


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(
            objective='binary',
            metric='auc',
            boosting_type='gbdt',
            num_leaves=trial.suggest_int('num_leaves', 16, 256),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            lambda_l1=trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
            lambda_l2=trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
            feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
            bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
            bagging_freq=trial.suggest_int('bagging_freq', 1, 7),
            min_child_samples=trial.suggest_int('min_child_samples', 10, 100),
            verbosity=-1,
            n_estimators=2000,
            random_state=42,
            n_jobs=1
        )
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-25 18:28:50,709] A new study created in memory with name: no-name-40bf84bf-ed96-466b-ab45-97c30fe732b1
Best trial: 6. Best value: 0.948279:   2%|▊                                         | 1/50 [06:45<5:31:01, 405.33s/it]

[I 2026-05-25 18:35:36,035] Trial 6 finished with value: 0.9482785396987762 and parameters: {'num_leaves': 21, 'max_depth': 8, 'learning_rate': 0.07766962337877835, 'lambda_l1': 0.29626552967323505, 'lambda_l2': 0.03904837453366228, 'feature_fraction': 0.7951296479075001, 'bagging_fraction': 0.7347101913928169, 'bagging_freq': 3, 'min_child_samples': 98}. Best is trial 6 with value: 0.9482785396987762.


Best trial: 10. Best value: 0.949829:   4%|█▋                                       | 2/50 [07:02<2:21:30, 176.88s/it]

[I 2026-05-25 18:35:53,005] Trial 10 finished with value: 0.9498289448758748 and parameters: {'num_leaves': 56, 'max_depth': 4, 'learning_rate': 0.03138650974606845, 'lambda_l1': 0.004123567265718793, 'lambda_l2': 0.2444920460453599, 'feature_fraction': 0.7418366295967, 'bagging_fraction': 0.8487509229313378, 'bagging_freq': 6, 'min_child_samples': 36}. Best is trial 10 with value: 0.9498289448758748.


Best trial: 5. Best value: 0.950051:   6%|██▌                                       | 3/50 [07:17<1:20:44, 103.08s/it]

...

[I 2026-05-25 19:20:30,346] Trial 47 finished with value: 0.9500804528494594 and parameters: {'num_leaves': 63, 'max_depth': 3, 'learning_rate': 0.019521485982696016, 'lambda_l1': 0.016138439255424266, 'lambda_l2': 0.038514643456939175, 'feature_fraction': 0.71638455279854, 'bagging_fraction': 0.7830358418451041, 'bagging_freq': 5, 'min_child_samples': 79}. Best is trial 32 with value: 0.9501301405647788.


Best trial: 32. Best value: 0.95013:  98%|███████████████████████████████████████████ | 49/50 [52:31<00:32, 32.25s/it]

[I 2026-05-25 19:21:22,591] Trial 49 finished with value: 0.9499930587542377 and parameters: {'num_leaves': 64, 'max_depth': 4, 'learning_rate': 0.020010676587010726, 'lambda_l1': 0.013090534904234183, 'lambda_l2': 0.03847833586768308, 'feature_fraction': 0.7656716747084156, 'bagging_fraction': 0.7851112167005444, 'bagging_freq': 5, 'min_child_samples': 81}. Best is trial 32 with value: 0.9501301405647788.


Best trial: 32. Best value: 0.95013: 100%|████████████████████████████████████████████| 50/50 [52:54<00:00, 63.50s/it]

[I 2026-05-25 19:21:45,480] Trial 48 finished with value: 0.9500792287935713 and parameters: {'num_leaves': 64, 'max_depth': 4, 'learning_rate': 0.013828073122423496, 'lambda_l1': 0.01567643521531724, 'lambda_l2': 0.058055200886465506, 'feature_fraction': 0.7699927125970286, 'bagging_fraction': 0.883815805932063, 'bagging_freq': 3, 'min_child_samples': 79}. Best is trial 32 with value: 0.9501301405647788.
Best trial score:
0.9501301405647788

Best params:
{'num_leaves': 17, 'max_depth': 3, 'learning_rate': 0.01004043509941534, 'lambda_l1': 0.0014302643904862981, 'lambda_l2': 0.012449043608801802, 'feature_fraction': 0.6996516891512191, 'bagging_fraction': 0.8453819807346918, 'bagging_freq': 4, 'min_child_samples': 65}


In [12]:
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=100, n_jobs=-1, show_progress_bar=True)

Best trial: 32. Best value: 0.95013:   1%|▍                                       | 1/100 [10:10<16:46:39, 610.10s/it]

[I 2026-05-25 19:51:40,112] Trial 56 finished with value: 0.9500279557859661 and parameters: {'num_leaves': 34, 'max_depth': 5, 'learning_rate': 0.012909756801026752, 'lambda_l1': 0.0061762716742159805, 'lambda_l2': 0.09464335077426987, 'feature_fraction': 0.7484448465030112, 'bagging_fraction': 0.8308398269500364, 'bagging_freq': 5, 'min_child_samples': 47}. Best is trial 32 with value: 0.9501301405647788.


Best trial: 32. Best value: 0.95013:   2%|▊                                        | 2/100 [10:20<7:00:42, 257.58s/it]

[I 2026-05-25 19:51:50,924] Trial 57 finished with value: 0.9500254288888705 and parameters: {'num_leaves': 36, 'max_depth': 5, 'learning_rate': 0.013111932582038483, 'lambda_l1': 0.005624223535175666, 'lambda_l2': 0.14460144527437616, 'feature_fraction': 0.7450531906159246, 'bagging_fraction': 0.8298649801168149, 'bagging_freq': 5, 'min_child_samples': 48}. Best is trial 32 with value: 0.9501301405647788.


Best trial: 32. Best value: 0.95013:   3%|█▏                                       | 3/100 [10:23<3:48:28, 141.32s/it]

...

[I 2026-05-25 20:55:28,167] Trial 147 finished with value: 0.9501244617658923 and parameters: {'num_leaves': 195, 'max_depth': 3, 'learning_rate': 0.012565289999093126, 'lambda_l1': 0.34268952053104595, 'lambda_l2': 1.1904159905642004, 'feature_fraction': 0.7723902397419522, 'bagging_fraction': 0.7750639263342847, 'bagging_freq': 4, 'min_child_samples': 64}. Best is trial 125 with value: 0.9501335417861044.


Best trial: 125. Best value: 0.950134:  99%|██████████████████████████████████████▌| 99/100 [1:14:03<00:16, 16.48s/it]

[I 2026-05-25 20:55:33,127] Trial 148 finished with value: 0.9501186224519005 and parameters: {'num_leaves': 128, 'max_depth': 3, 'learning_rate': 0.012545542730901164, 'lambda_l1': 0.002720039839131347, 'lambda_l2': 0.716508673802421, 'feature_fraction': 0.8796832719969072, 'bagging_fraction': 0.7790773799001411, 'bagging_freq': 4, 'min_child_samples': 69}. Best is trial 125 with value: 0.9501335417861044.


Best trial: 125. Best value: 0.950134: 100%|██████████████████████████████████████| 100/100 [1:14:20<00:00, 44.60s/it]

[I 2026-05-25 20:55:50,150] Trial 149 finished with value: 0.9501259116763784 and parameters: {'num_leaves': 129, 'max_depth': 3, 'learning_rate': 0.012588066259379983, 'lambda_l1': 0.10189569299996734, 'lambda_l2': 0.04739755566387959, 'feature_fraction': 0.7726157151574102, 'bagging_fraction': 0.7787755416846514, 'bagging_freq': 4, 'min_child_samples': 64}. Best is trial 125 with value: 0.9501335417861044.


In [14]:
stacking_lgbm = LGBMClassifier(
    **study.best_params, 
    verbosity=-1, 
    n_estimators=2000,
    random_state=42,
    objective='binary',
    metric='auc',
    boosting_type='gbdt'
).fit(X_train, y_train.PitNextLap)

# Submission

In [15]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [16]:
submission['PitNextLap'] = stacking_lgbm.predict_proba(X_test)[:, 1]

In [17]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [18]:
X_train.columns

Index(['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba',
       'pca_knn_proba', 'ridge_logit', 'truncatedsvd_linear_svc_logit',
       'truncatedsvd_ridge_logit', 'sgdclassifier_proba',
       'truncatedsvd_lgbm_proba', 'truncatedsvd_knn_proba',
       'truncatedsvd_nystroem_knn_proba', 'truncatedsvd_rbfsampler_knn_proba',
       'truncatedsvd_logistic_regression_proba',
       'truncatedsvd_nystroem_logistic_regression_proba',
       'truncatedsvd_nystroem_sgdclassifier_proba',
       'truncatedsvd_rbfsampler_logistic_regression_proba',
       'truncatedsvd_rbfsampler_sgdclassifier_proba',
       'voting_lgbm_and_cat_and_xgb_and_hist_proba',
       'voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier',
       'voting_truncatedsvd_nytroem_knn_and_